# 2.3. Exploración de los Datos

Fase 2.3 de CRISP-DM: carga, inspección y visualización del dataset para encontrar patrones, estructuras y relaciones entre variables. Este notebook cubre las subsecciones 2.3.1 (Carga), 2.3.2 (Inspección) y 2.3.3 (Visualización).

## 2.3.1 Carga de los datos

In [ ]:
import json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '../data'
CSV_PATH = f'{DATA_DIR}/GBvideos_cc50_202101.csv'
JSON_PATH = f'{DATA_DIR}/GB_category_id.json'
FIG_DIR = '../reports/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Carga del CSV y mapeo de categorías
df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    categories_data = json.load(f)
category_map = {int(i['id']): i['snippet']['title'] for i in categories_data['items']}
df['category_name'] = df['category_id'].map(category_map).fillna('Unknown')

# Conversión temporal
df['publish_time'] = pd.to_datetime(df['publish_time'], errors='coerce')
df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')

# Variables derivadas básicas para el análisis
df['engagement_rate'] = (df['likes'] + df['dislikes'] + df['comment_count']) / (df['views'] + 1)
df['likes_per_view'] = df['likes'] / (df['views'] + 1)
df['comments_per_view'] = df['comment_count'] / (df['views'] + 1)
df['like_dislike_ratio'] = df['likes'] / (df['dislikes'] + 1)
df['publish_weekday'] = df['publish_time'].dt.weekday
df['publish_hour'] = df['publish_time'].dt.hour
print('Shape:', df.shape)

## 2.3.2 Inspección de datos

In [ ]:
display(df.head())
display(df.info())
print('Columnas:', df.columns.tolist())

## 2.3.3 Visualización de datos

### 2.3.3.1 Análisis univariado — numérico

In [ ]:
numeric_cols = ['views', 'likes', 'dislikes', 'comment_count']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.histplot(np.log1p(df[col]), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribución de {col} (log1p)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/univariate_histograms.png', dpi=300, bbox_inches='tight'); plt.show()

for col in numeric_cols:
    print(f'{col}: skew={df[col].skew():.2f} | kurtosis={df[col].kurtosis():.2f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.boxplot(x=np.log1p(df[col]), ax=ax, color='coral')
    ax.set_title(f'Boxplot de {col} (log1p)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/univariate_boxplots.png', dpi=300, bbox_inches='tight'); plt.show()

### 2.3.3.2 Análisis univariado — categórico

In [ ]:
counts = df['category_name'].value_counts().head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=counts.values, y=counts.index, palette='viridis', hue=counts.index, legend=False)
plt.title('Top 15 categorías en tendencias (GB)'); plt.xlabel('Frecuencia')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/top_categories.png', dpi=300, bbox_inches='tight'); plt.show()
display(counts.to_frame('frecuencia'))

### 2.3.3.3 Análisis univariado — booleano

In [ ]:
for col in ['comments_disabled', 'ratings_disabled', 'video_error_or_removed']:
    if col in df.columns:
        print(df[col].value_counts(normalize=True))

### 2.3.3.4 Análisis temporal

In [ ]:
serie = df.groupby(df['trending_date'].dt.to_period('W'))['views'].mean().reset_index()
serie['trending_date'] = serie['trending_date'].dt.to_timestamp()
plt.figure(figsize=(12, 5))
sns.lineplot(data=serie, x='trending_date', y='views', color='navy')
plt.title('Evolución semanal de vistas promedio')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/time_series_views.png', dpi=300, bbox_inches='tight'); plt.show()

if 'publish_weekday' in df.columns:
    pivot = df.groupby('publish_weekday')['views'].mean()
    display(pivot.to_frame('views_promedio'))

### 2.3.3.5 Correlaciones

In [ ]:
corr_cols = ['views', 'likes', 'dislikes', 'comment_count',
             'engagement_rate', 'likes_per_view', 'comments_per_view']
corr = df[corr_cols].corr(method='pearson')
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación (Pearson)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/correlation_pearson.png', dpi=300, bbox_inches='tight'); plt.show()

### 2.3.3.6 Relaciones bivariadas

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(x=np.log1p(df['likes']), y=np.log1p(df['views']),
            scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'}, color='teal')
plt.xlabel('log1p(likes)'); plt.ylabel('log1p(views)'); plt.title('Views vs Likes (log1p)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/views_vs_likes.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(x=np.log1p(df['comment_count']), y=np.log1p(df['views']),
            scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'}, color='darkorange')
plt.xlabel('log1p(comment_count)'); plt.ylabel('log1p(views)'); plt.title('Views vs Comentarios (log1p)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/views_vs_comments.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
top_cats = df['category_name'].value_counts().head(10).index
data = df[df['category_name'].isin(top_cats)].copy()
data['log_views'] = np.log1p(data['views'])
plt.figure(figsize=(12, 6))
sns.boxplot(data=data, x='category_name', y='log_views', palette='Set2',
            hue='category_name', legend=False)
plt.xticks(rotation=45, ha='right'); plt.title('Vistas por categoría (log1p)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/views_by_category.png', dpi=300, bbox_inches='tight'); plt.show()

### 2.3.3.7 Análisis multivariado

In [ ]:
selected = [c for c in ['views', 'likes', 'comment_count', 'engagement_rate'] if c in df.columns]
sns.pairplot(np.log1p(df[selected]))

## Guardado del dataset para la siguiente fase

In [ ]:
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
CLEAN_CSV = f'{DATA_DIR}/processed/gb_videos_clean.csv'
df.to_csv(CLEAN_CSV, index=False)
print('Dataset limpio guardado en', CLEAN_CSV)